# Extended Kalman Filter (EKF): High-Level Overview

Before diving into the math, let's understand **what** we are computing and **why**.

### The Problem: Sensor Fusion
We want to know the 3D orientation (tilt/rotation) of a device (e.g., a drone or earbud). We have two sensors:
1.  **Gyroscope:** Measures rotation speed. It is very fast and smooth, but it **drifts** (errors accumulate over time, like walking with your eyes closed).
2.  **Accelerometer:** Measures gravity. It knows "which way is down," so it doesn't drift. However, it is **noisy** and gets confused if you shake the device.

### The Solution: The 2-Step Loop
The EKF is a loop that runs hundreds of times per second:
1.  **Predict (The Gyro Step):** We assume the Gyroscope is correct for a split second. We perform **Matrix Multiplication** to rotate our current state forward in time.
2.  **Correct (The Accel Step):** We check the Accelerometer. If our prediction says "Gravity is Up" but the Accelerometer says "Gravity is Down," we calculate a **weighted average** (Kalman Gain) to fix our mistake. This involves **Matrix Inversion**.

### Computational Profile
* **Prediction:** Heavy on Matrix Multiplication ($7 \times 7$ matrices).
* **Correction:** Heavy on Linear Algebra (Inverting matrices to solve equations).

---

# Mathematical Walkthrough

**Objective:** Estimate the State Vector $x$ (7 variables) using noisy measurements.

**Notation Key:**
* $x_k$: State vector at time step $k$ (7x1)
* $P_k$: Error Covariance Matrix (7x7) -- *"How unsure are we?"*
* $F_k$: State Transition Jacobian (7x7) -- *"How does the state change?"*
* $H_k$: Measurement Jacobian (3x7) -- *"How do sensors relate to state?"*
* $K_k$: Kalman Gain (7x3) -- *"How much should we trust the sensor?"*
* $q$: Quaternion $[q_w, q_x, q_y, q_z]$ -- *3D Rotation format*


In [ ]:
import numpy as np
import csv
import matplotlib.pyplot as plt

# Configuration
DT_DEFAULT = 0.005  # Sampling time (200Hz)


## 1. State Initialization

**Reasoning:** The filter needs a starting guess. We assume the device starts flat (Identity Quaternion) and stationary.
**Output:** $x$ (7D Vector), $P$ (7x7 Matrix)
**Meaning:** $x$ holds our current "best guess" of reality. $P$ holds our "uncertainty" about that guess.

---

State Vector $x$ contains 7 elements:
1. **Orientation Quaternion** $q$ (4 elements): $[q_w, q_x, q_y, q_z]$
2. **Gyroscope Bias** $b$ (3 elements): $[b_x, b_y, b_z]$ (To correct sensor drift)

$$ x = \begin{bmatrix} q_w \\ q_x \\ q_y \\ q_z \\ b_x \\ b_y \\ b_z \end{bmatrix} $$

In [ ]:
# 1. Initialize State
# Identity Quaternion [1, 0, 0, 0] + Zero Bias [0, 0, 0]
x = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

# 2. Initialize Uncertainty P
# Diagonal matrix with small initial uncertainty
P = np.eye(7) * 0.1

# 3. Noise Matrices
Q = np.eye(7) * 1e-4  # Process Noise (Trust model mostly)
R = np.eye(3) * 0.05  # Measurement Noise (Trust accel moderately)

## 2. Prediction Step (Time Update)

**Plain English:** We take the Gyroscope reading (speed of rotation) and mathematically "spin" our quaternion forward by `dt` seconds. We also increase our uncertainty $P$ slightly because physics models are never perfect.

**Computation:** This step is dominated by **Matrix-Vector Multiplication**. We multiply the $4 \times 4$ rotation matrix $\Omega$ by the state.

---

### A. Physics Model (Integration)
We integrate angular velocity $\omega$ to update the quaternion.
$$ \dot{q} = \frac{1}{2} \Omega(\omega) q $$
$$ x_{k|k-1} = x_{k-1} + \dot{q} \cdot \Delta t $$

### B. Jacobian $F$ (Linearization)
Since quaternions are non-linear, we linearize the system to update the Covariance $P$.
$$ P_{k|k-1} = F_k P_{k-1} F_k^T + Q $$

In [ ]:
def predict(x, P, gyro, dt, Q):
    q = x[0:4]
    bias = x[4:7]
    
    # Remove estimated bias from raw gyro measurement
    w = gyro - bias
    wx, wy, wz = w[0], w[1], w[2]
    
    # Quaternion Omega Matrix
    Omega = np.array([
        [0, -wx, -wy, -wz],
        [wx, 0, wz, -wy],
        [wy, -wz, 0, wx],
        [wz, wy, -wx, 0]
    ])
    
    # 1. State Prediction (Integration)
    dq = 0.5 * np.dot(Omega, q) * dt
    x_pred = x.copy()
    x_pred[0:4] += dq
    
    # Normalize Quaternion (Constraint ||q||=1)
    x_pred[0:4] /= np.linalg.norm(x_pred[0:4])
    
    # 2. Jacobian F Construction
    F = np.eye(7)
    # Top-Left: Rotation influence (I + 0.5*Omega*dt)
    F[0:4, 0:4] += 0.5 * Omega * dt
    
    # Top-Right: Bias influence (Complex derivation simplified)
    q0, q1, q2, q3 = q
    G_bias = 0.5 * dt * np.array([
        [q1, q2, q3],
        [-q0, q3, -q2],
        [-q3, -q0, q1],
        [q2, -q1, -q0]
    ])
    F[0:4, 4:7] = G_bias
    
    # 3. Covariance Prediction
    P_pred = np.dot(np.dot(F, P), F.T) + Q
    
    return x_pred, P_pred

# Dummy Step
dummy_gyro = np.array([0.1, 0.0, 0.0]) # Rotating X-axis
x, P = predict(x, P, dummy_gyro, DT_DEFAULT, Q)
print(f"Predicted Quaternion: {x[0:4]}")

## 3. Update Step (Measurement Correction)

**Plain English:** We look at the Accelerometer. If we are perfectly upright, it should read $[0, 0, 9.8]$. If it reads something else, we calculate the "error" (Residual). Then, we perform a heavy calculation to find the "Kalman Gain" ($K$), which tells us how much to shift our quaternion to match gravity.

**Computation:** This step is computationally expensive because it requires **Matrix Inversion** (`np.linalg.inv`) to solve for $K$.

---

### A. Predicted Measurement $h(x)$
What should the accelerometer see if our orientation $q$ is correct? It should see Gravity vector $g=[0,0,1]$ rotated into the device frame.
$$ h(x) = R(q)^T \cdot \begin{bmatrix} 0 \\ 0 \\ 1 \end{bmatrix} $$

### B. Residual $y$
The difference between what we see (Accel) and what we expected (Gravity).
$$ y = z_{meas} - h(x) $$

### C. Kalman Gain $K$
We compute $K$ to decide how much to trust the measurement vs. our prediction.
$$ S = H P H^T + R $$
$$ K = P H^T S^{-1} $$

In [ ]:
def update(x, P, accel, R):
    q = x[0:4]
    q0, q1, q2, q3 = q
    
    # 1. Predicted Gravity (h(x))
    # Last row of Rotation Matrix
    g_pred = np.array([
        2*(q1*q3 - q0*q2),
        2*(q0*q1 + q2*q3),
        q0**2 - q1**2 - q2**2 + q3**2
    ])
    
    # Normalize real measurement
    accel_norm = np.linalg.norm(accel)
    if accel_norm == 0: return x, P
    z_meas = accel / accel_norm
    
    # 2. Residual y
    y = z_meas - g_pred
    
    # 3. Jacobian H (3x7)
    # Derivative of g_pred w.r.t q
    H = np.zeros((3, 7))
    H[0,0], H[0,1], H[0,2], H[0,3] = -2*q2,  2*q3, -2*q0, 2*q1
    H[1,0], H[1,1], H[1,2], H[1,3] =  2*q1,  2*q0,  2*q3, 2*q2
    H[2,0], H[2,1], H[2,2], H[2,3] =  2*q0, -2*q1, -2*q2, 2*q3
    
    # 4. Kalman Gain K
    # S = H P H^T + R
    S = np.dot(np.dot(H, P), H.T) + R
    # K = P H^T inv(S)
    K = np.dot(np.dot(P, H.T), np.linalg.inv(S))
    
    # 5. Correction
    dx = np.dot(K, y)
    x_new = x + dx
    x_new[0:4] /= np.linalg.norm(x_new[0:4]) # Re-normalize
    
    # 6. Covariance Update
    I = np.eye(7)
    P_new = np.dot((I - np.dot(K, H)), P)
    
    return x_new, P_new

# Dummy Update
dummy_accel = np.array([0.0, 0.0, 9.81]) # Gravity down
x, P = update(x, P, dummy_accel, R)
print(f"Corrected Quaternion: {x[0:4]}")

## 4. Full Pipeline Loop

We loop through the EuRoC dataset. Note the logic check: we only run the **Update** step if the accelerometer magnitude is close to 9.81. If the drone is accelerating hard (e.g., taking off), the accelerometer measures Motion + Gravity, which would confuse the filter, so we skip the update.

In [ ]:
def run_pipeline(imu_data, x, P):
    trajectory = []
    
    for sample in imu_data:
        dt = sample['dt']
        gyro = sample['gyro']
        accel = sample['accel']
        
        # 1. Predict
        x, P = predict(x, P, gyro, dt, Q)
        
        # 2. Update (Conditional)
        acc_mag = np.linalg.norm(accel)
        if abs(acc_mag - 9.81) < 2.0:
            x, P = update(x, P, accel, R)
            
        trajectory.append(x[0:4].copy())
        
    return np.array(trajectory)

# Mock Data for Notebook demonstration
mock_data = [{'dt': 0.01, 'gyro': np.array([0,0,0.1]), 'accel': np.array([0,0,9.81])} for _ in range(100)]
traj = run_pipeline(mock_data, x, P)
print(f"Generated {len(traj)} trajectory points.")